In [2]:
import logging
import os
import sys
sys.path.append("../")
import glob
import numpy as np
import torch
from tqdm import tqdm
#from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import monai
from monai.data import ImageDataset, DataLoader
import monai.transforms as transforms
from monai.transforms import EnsureChannelFirst, Compose, RandRotate90, Resize, ScaleIntensity
import nibabel as nib
import pandas as pd

from utils.custom_transforms import ScaleIntensityFromHistogramPeak, SetBackgroundToZero, SelectChannelsd

In [3]:
#ROOT_DIR = "/home/fehrdelt/bettik/"
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"

# SOOP

### Clinical data (class label) 1: nihss>15, 0: nihss<=15

In [40]:
participants_tsv_path = ROOT_DIR+"datasets/final_soop_dataset_small/participants.tsv"

participants_soop_df = pd.read_csv(participants_tsv_path, sep="\t")

In [41]:
participants_soop_df.head()

,participant_id,sex,age,race,acuteischaemicstroke,priorstroke,bmi,nihss,gs_rankin_6isdeath
0,sub-2,M,78.0,w,1.0,0.0,22.84,17.0,NaN
1,sub-3,F,87.0,w,1.0,1.0,19.23,15.0,6.0
2,sub-5,M,58.0,w,1.0,0.0,37.29,1.0,NaN
3,sub-6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sub-7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# drop rows where 'nihss' is NaN
participants_soop_df = participants_soop_df.dropna(subset=["nihss"])

In [43]:
participants_soop_df.head()

,participant_id,sex,age,race,acuteischaemicstroke,priorstroke,bmi,nihss,gs_rankin_6isdeath
0,sub-2,M,78.0,w,1.0,0.0,22.84,17.0,NaN
1,sub-3,F,87.0,w,1.0,1.0,19.23,15.0,6.0
2,sub-5,M,58.0,w,1.0,0.0,37.29,1.0,NaN
5,sub-8,F,34.0,b,1.0,0.0,23.14,19.0,NaN
6,sub-9,F,70.0,w,1.0,0.0,26.89,18.0,4.0


In [44]:
participants_soop_df["high_nihss"] = (participants_soop_df["nihss"] > 15).astype(np.int64)

participants_soop_df.head()

,participant_id,sex,age,race,acuteischaemicstroke,priorstroke,bmi,nihss,gs_rankin_6isdeath,high_nihss
0,sub-2,M,78.0,w,1.0,0.0,22.84,17.0,NaN,1
1,sub-3,F,87.0,w,1.0,1.0,19.23,15.0,6.0,0
2,sub-5,M,58.0,w,1.0,0.0,37.29,1.0,NaN,0
5,sub-8,F,34.0,b,1.0,0.0,23.14,19.0,NaN,1
6,sub-9,F,70.0,w,1.0,0.0,26.89,18.0,4.0,1


In [45]:
counts = participants_soop_df["high_nihss"].value_counts()
count_high_nihss_1 = int(counts.get(1, 0))
count_high_nihss_0 = int(counts.get(0, 0))

print(f"high_nihss==1: {count_high_nihss_1}")
print(f"high_nihss==0: {count_high_nihss_0}")

high_nihss==1: 209
high_nihss==0: 897


#### Drop rows where high_nihss == 0 (minority class undersampling)
TODO: remove and do this properly with scikit-learn pipeline and SMOTE if doing gradient boosting (fastdiag style)

In [46]:
# Undersample class 0 so both classes have equal counts
counts = participants_soop_df["high_nihss"].value_counts()
count_high_nihss_1 = int(counts.get(1, 0))
count_high_nihss_0 = int(counts.get(0, 0))

# Number of class-0 rows to remove to match class-1 count
n_drop = max(0, count_high_nihss_0 - count_high_nihss_1)

if n_drop > 0:
    drop_idx = participants_soop_df[participants_soop_df["high_nihss"] == 0] \
        .sample(n=n_drop, random_state=42).index
    participants_soop_df = participants_soop_df.drop(drop_idx).reset_index(drop=True)

# Recompute and display final counts
counts = participants_soop_df["high_nihss"].value_counts()
count_high_nihss_1 = int(counts.get(1, 0))
count_high_nihss_0 = int(counts.get(0, 0))

print("After balancing:")
print(f"high_nihss==1: {count_high_nihss_1}")
print(f"high_nihss==0: {count_high_nihss_0}")

After balancing:
high_nihss==1: 209
high_nihss==0: 209


In [47]:
# Stratified split of participants_soop_df into train/val/test
# Fractions can be adjusted if needed
train_frac, val_frac, test_frac = 0.7, 0.15, 0.15
SPLIT_SEED = 42

assert "high_nihss" in participants_soop_df.columns, "Column 'high_nihss' not found."

rng = np.random.default_rng(SPLIT_SEED)
train_idx, val_idx, test_idx = [], [], []

for cls in participants_soop_df["high_nihss"].unique():
    cls_idx = participants_soop_df.index[participants_soop_df["high_nihss"] == cls].to_numpy()
    cls_idx = rng.permutation(cls_idx)

    n = len(cls_idx)
    n_train = int(np.floor(n * train_frac))
    n_val = int(np.floor(n * val_frac))
    n_test = n - n_train - n_val

    train_idx.extend(cls_idx[:n_train])
    val_idx.extend(cls_idx[n_train:n_train + n_val])
    test_idx.extend(cls_idx[n_train + n_val:])

# Build dataframes
train_participants_soop_df = participants_soop_df.loc[train_idx].reset_index(drop=True)
val_participants_soop_df = participants_soop_df.loc[val_idx].reset_index(drop=True)
test_participants_soop_df = participants_soop_df.loc[test_idx].reset_index(drop=True)

def _dist(df):
    vc = df["high_nihss"].value_counts()
    return f"total={len(df)}, class0={int(vc.get(0,0))}, class1={int(vc.get(1,0))}"

print("Split summary:")
print(f"Train: {_dist(train_participants_soop_df)}")
print(f"Val:   {_dist(val_participants_soop_df)}")
print(f"Test:  {_dist(test_participants_soop_df)}")

Split summary:
Train: total=292, class0=146, class1=146
Val:   total=62, class0=31, class1=31
Test:  total=64, class0=32, class1=32


In [48]:
train_participants_soop_df.to_csv(ROOT_DIR+"StrokeUADiag/data_splits_lists/soop/train_participants.csv", index=False)
val_participants_soop_df.to_csv(ROOT_DIR+"StrokeUADiag/data_splits_lists/soop/val_participants.csv", index=False)
test_participants_soop_df.to_csv(ROOT_DIR+"StrokeUADiag/data_splits_lists/soop/test_participants.csv", index=False)

# AINI-Stroke

### Clinical data (class label) 1: nihss>15, 0: nihss<=15

In [49]:
clinical_data_2022_path = ROOT_DIR+"datasets/aini-stroke_clinical_data/clinical_data_2022_with_shanoir_name.csv"
clinical_data_2023_path = ROOT_DIR+"datasets/aini-stroke_clinical_data/clinical_data_2023_with_shanoir_name.csv"

clinical_data_2022_df = pd.read_csv(clinical_data_2022_path)
clinical_data_2023_df = pd.read_csv(clinical_data_2023_path)

# combine the two clinical dataframes
participants_ainistroke_df = pd.concat([clinical_data_2022_df, clinical_data_2023_df], ignore_index=True)

In [50]:
participants_ainistroke_df.head()

,n°patient,shanoir_name,Age,sexe,UNV,mode_entree_CHUGA,alerte_AVC,date_admission,heure_admission,date_sortie,...,date_thrombolyse,heure_thrombolyse,date_ponction,heure_ponction,date_recanalisation,heure_recanalisation,TICI_final,commentaire,fiche_validé,IPP
0,2022UNVG-3,aini-stroke-22001,87,F,Grenoble,via SAU,Oui,1/1/2022,2:42,1/21/2022,...,NaN,NaN,1/1/2022,4:48,1/1/2022,5:06,3,NaN,OUI,NaN
1,2022UNVG-4,aini-stroke-22002,95,F,Grenoble,via SAU,Oui,1/1/2022,20:51,1/20/2022,...,1/1/2022,22:08,NaN,NaN,NaN,NaN,NaN,NaN,OUI,NaN
2,2022UNVG-5,aini-stroke-22003,71,F,Grenoble,via SAU,Oui,1/1/2022,20:40,1/13/2022,...,1/1/2022,21:45,1/1/2022,22:32,1/1/2022,22:42,2b,NaN,OUI,NaN
3,2022UNVG-6,aini-stroke-22004,53,H,Grenoble,via SAU,Oui,1/1/2022,17:13,1/5/2022,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Episode atypique de paresthésies de l'hémilang...,OUI,NaN
4,2022UNVG-7,aini-stroke-22005,70,F,Grenoble,via SAU,Non,1/2/2022,15:34,1/5/2022,...,1/2/2022,17:38,NaN,NaN,NaN,NaN,NaN,"Hypoperfusion cervelet, pas de lession à l'IRM",OUI,NaN


In [51]:
participants_ainistroke_df["NIHSS_initial"]

0       17.0
1       18.0
2       16.0
3        NaN
4        0.0
        ... 
1679    15.0
1680    15.0
1681     0.0
1682     0.0
1683     NaN
Name: NIHSS_initial, Length: 1684, dtype: float64

In [52]:
# drop rows where 'nihss' is NaN
participants_ainistroke_df = participants_ainistroke_df.dropna(subset=["NIHSS_initial"])

In [53]:
participants_ainistroke_df["high_nihss"] = (participants_ainistroke_df["NIHSS_initial"] > 15).astype(np.int64)

participants_ainistroke_df.head()

,n°patient,shanoir_name,Age,sexe,UNV,mode_entree_CHUGA,alerte_AVC,date_admission,heure_admission,date_sortie,...,heure_thrombolyse,date_ponction,heure_ponction,date_recanalisation,heure_recanalisation,TICI_final,commentaire,fiche_validé,IPP,high_nihss
0,2022UNVG-3,aini-stroke-22001,87,F,Grenoble,via SAU,Oui,1/1/2022,2:42,1/21/2022,...,NaN,1/1/2022,4:48,1/1/2022,5:06,3,NaN,OUI,NaN,1
1,2022UNVG-4,aini-stroke-22002,95,F,Grenoble,via SAU,Oui,1/1/2022,20:51,1/20/2022,...,22:08,NaN,NaN,NaN,NaN,NaN,NaN,OUI,NaN,1
2,2022UNVG-5,aini-stroke-22003,71,F,Grenoble,via SAU,Oui,1/1/2022,20:40,1/13/2022,...,21:45,1/1/2022,22:32,1/1/2022,22:42,2b,NaN,OUI,NaN,1
4,2022UNVG-7,aini-stroke-22005,70,F,Grenoble,via SAU,Non,1/2/2022,15:34,1/5/2022,...,17:38,NaN,NaN,NaN,NaN,NaN,"Hypoperfusion cervelet, pas de lession à l'IRM",OUI,NaN,0
5,2022UNVG-8,aini-stroke-22006,60,H,Grenoble,via IRM directe,Oui,1/3/2022,10:59,1/31/2022,...,NaN,1/3/2022,12:50,1/3/2022,13:51,NaN,TICI =donnée manquante,OUI,NaN,1


In [54]:
counts = participants_ainistroke_df["high_nihss"].value_counts()
count_high_nihss_1 = int(counts.get(1, 0))
count_high_nihss_0 = int(counts.get(0, 0))

print(f"high_nihss==1: {count_high_nihss_1}")
print(f"high_nihss==0: {count_high_nihss_0}")

high_nihss==1: 232
high_nihss==0: 1295


#### Drop rows where high_nihss == 0 (minority class undersampling)
TODO: remove and do this properly with scikit-learn pipeline and SMOTE if doing gradient boosting (fastdiag style)

In [55]:
# Undersample class 0 so both classes have equal counts
counts = participants_ainistroke_df["high_nihss"].value_counts()
count_high_nihss_1 = int(counts.get(1, 0))
count_high_nihss_0 = int(counts.get(0, 0))

# Number of class-0 rows to remove to match class-1 count
n_drop = max(0, count_high_nihss_0 - count_high_nihss_1)

if n_drop > 0:
    drop_idx = participants_ainistroke_df[participants_ainistroke_df["high_nihss"] == 0] \
        .sample(n=n_drop, random_state=42).index
    participants_ainistroke_df = participants_ainistroke_df.drop(drop_idx).reset_index(drop=True)

# Recompute and display final counts
counts = participants_ainistroke_df["high_nihss"].value_counts()
count_high_nihss_1 = int(counts.get(1, 0))
count_high_nihss_0 = int(counts.get(0, 0))

print("After balancing:")
print(f"high_nihss==1: {count_high_nihss_1}")
print(f"high_nihss==0: {count_high_nihss_0}")

After balancing:
high_nihss==1: 232
high_nihss==0: 232


In [56]:
# Stratified split of participants_ainistroke_df into train/val/test
# Fractions can be adjusted if needed
train_frac, val_frac, test_frac = 0.7, 0.15, 0.15
SPLIT_SEED = 42

assert "high_nihss" in participants_ainistroke_df.columns, "Column 'high_nihss' not found."

rng = np.random.default_rng(SPLIT_SEED)
train_idx, val_idx, test_idx = [], [], []

for cls in participants_ainistroke_df["high_nihss"].unique():
    cls_idx = participants_ainistroke_df.index[participants_ainistroke_df["high_nihss"] == cls].to_numpy()
    cls_idx = rng.permutation(cls_idx)

    n = len(cls_idx)
    n_train = int(np.floor(n * train_frac))
    n_val = int(np.floor(n * val_frac))
    n_test = n - n_train - n_val

    train_idx.extend(cls_idx[:n_train])
    val_idx.extend(cls_idx[n_train:n_train + n_val])
    test_idx.extend(cls_idx[n_train + n_val:])

# Build dataframes
train_participants_ainistroke_df = participants_ainistroke_df.loc[train_idx].reset_index(drop=True)
val_participants_ainistroke_df = participants_ainistroke_df.loc[val_idx].reset_index(drop=True)
test_participants_ainistroke_df = participants_ainistroke_df.loc[test_idx].reset_index(drop=True)

def _dist(df):
    vc = df["high_nihss"].value_counts()
    return f"total={len(df)}, class0={int(vc.get(0,0))}, class1={int(vc.get(1,0))}"

print("Split summary:")
print(f"Train: {_dist(train_participants_ainistroke_df)}")
print(f"Val:   {_dist(val_participants_ainistroke_df)}")
print(f"Test:  {_dist(test_participants_ainistroke_df)}")

Split summary:
Train: total=324, class0=162, class1=162
Val:   total=68, class0=34, class1=34
Test:  total=72, class0=36, class1=36


In [58]:
train_participants_ainistroke_df.to_csv(ROOT_DIR+"StrokeUADiag/data_splits_lists/aini_stroke/train_participants.csv", index=False)
val_participants_ainistroke_df.to_csv(ROOT_DIR+"StrokeUADiag/data_splits_lists/aini_stroke/val_participants.csv", index=False)
test_participants_ainistroke_df.to_csv(ROOT_DIR+"StrokeUADiag/data_splits_lists/aini_stroke/test_participants.csv", index=False)